IIR 3D by optimization

---

Kishore Kumar Tarafdar, 25 May 2025

        Disable GPU: Force tensorflow to select CPU

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="-1"    
import tensorflow as tf

2025-06-19 07:58:49.152086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750300129.175950 4180163 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750300129.182892 4180163 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-19 07:58:49.207095: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


        Select a GPU with a memory limit

In [1]:
#%% # include ../dirx 
import tensorflow as tf
print(f"TensorFlow version {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
len(gpus)
mylibpath = [
    '/home/kishoretarafdar/bin',
    # '/data1/kishoretarafdar/src2D/GIIR/mra'
    #'/home/k/PLAYGROUND10GB/SKULSTRIPpaper__'
    ]
import sys
[sys.path.insert(1,_) for _ in mylibpath]
del mylibpath

from tf_select_a_gpu import select_a_gpu
# select_gpu = gpus[gpu_id]
memory_limit = 48#16#32 #GB
select_a_gpu(gpus, gpu_id = 2, memory_limit=memory_limit)
# del gpu_id, select_a_gpu, select_gpu

# from DWT1DFB import DWT1D, IDWT1D
# from DWT2DFB import DWT2D, IDWT2D
import matplotlib.pyplot as plt

2025-05-24 21:48:38.203653: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748103518.223497   34047 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748103518.229606   34047 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-24 21:48:38.249918: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version 2.18.0
Num GPUs Available:  3
3 Physical GPUs available 
Selected 1 Logical GPU with 48 GB memory limit


I0000 00:00:1748103520.336552   34047 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 49152 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


        3D shifts of x3D

        
$(\text{batch, N, N, N, inchannel}) \rightarrow (\text{batch, N, N, N, inchannel, outchannel, (Delays+1)}^3)$

In [2]:
import tensorflow as tf

def circular_shift_x_3d(x, delays, out_channels):
    """
    x: Tensor of shape (batch, N, N, N, channels)
    delays: int, number of delays in each dimension
    out_channels: int, number of output channels (identical copies)

    Returns:
        Tensor of shape (batch, N, N, N, channels, (D+1)^3, out_channels)
    """
    batch, N, _, _, channels = tf.unstack(tf.shape(x))

    # Create grid of 3D shifts: (D+1, D+1, D+1, 3) → (num_shifts, 3)
    shifts = tf.range(delays + 1)
    shift_grid = tf.stack(tf.meshgrid(shifts, shifts, shifts, indexing="ij"), axis=-1)  # (D+1, D+1, D+1, 3)
    shift_grid = tf.reshape(shift_grid, [-1, 3])  # (num_shifts, 3)

    # Broadcast x to (num_shifts, batch, N, N, N, channels)
    x_broadcasted = tf.expand_dims(x, axis=0)
    x_broadcasted = tf.repeat(x_broadcasted, repeats=tf.shape(shift_grid)[0], axis=0)

    # Roll each slice using vectorized shift
    def single_roll(args):
        x_i, shift = args
        return tf.roll(x_i, shift=[-shift[0], -shift[1], -shift[2]], axis=[1, 2, 3])

    rolled = tf.map_fn(
        single_roll,
        (x_broadcasted, shift_grid),
        fn_output_signature=tf.TensorSpec(shape=(None, None, None, None, None), dtype=x.dtype)
    )

    # rolled shape: (num_shifts, batch, N, N, N, channels) → (batch, N, N, N, channels, num_shifts)
    rolled = tf.transpose(rolled, perm=[1, 2, 3, 4, 5, 0])

    # Expand and tile to add out_channels: (batch, N, N, N, channels, num_shifts, out_channels)
    rolled = tf.expand_dims(rolled, axis=-1)
    rolled = tf.tile(rolled, [1, 1, 1, 1, 1, 1, out_channels])

    return rolled


In [16]:
x = tf.random.normal((2, 4, 4, 4, 3))
delays = 1
out_channels = 9

x_shifted = circular_shift_x_3d(x, delays, out_channels)
print(x_shifted.shape)  # (8, 32, 32, 4, 9, 3)

(2, 4, 4, 4, 3, 8, 9)


        3D shifts of y3D

$(\text{batch, N, N, N, inchannel, outchannel}) \rightarrow (\text{batch, N, N, N, inchannel, outchannel, (Delays+1)}^3)$

In [60]:
import tensorflow as tf

def circular_shift_y_3d(y, delays):
    """
    y: Tensor of shape (batch, N, N, N, channels, out_channels)
    delays: int, number of delays in each dimension

    Returns:
        Tensor of shape (batch, N, N, N, channels, (D+1)^3, out_channels)
    """
    batch, N, _, _, channels, out_channels = tf.unstack(tf.shape(y))

    # Create grid of 3D shifts: (D+1, D+1, D+1, 3) → (num_shifts, 3)
    shifts = tf.range(delays + 1)
    shift_grid = tf.stack(tf.meshgrid(shifts, shifts, shifts, indexing="ij"), axis=-1)  # (D+1, D+1, D+1, 3)
    shift_grid = tf.reshape(shift_grid, [-1, 3])  # (num_shifts, 3)

    # Broadcast y to (num_shifts, batch, N, N, N, channels, out_channels)
    y_broadcasted = tf.expand_dims(y, axis=0)
    y_broadcasted = tf.repeat(y_broadcasted, repeats=tf.shape(shift_grid)[0], axis=0)

    # Roll each slice using vectorized shift
    def single_roll(args):
        y_i, shift = args
        return tf.roll(y_i, shift=[-shift[0], -shift[1], -shift[2]], axis=[1, 2, 3])

    rolled = tf.map_fn(
        single_roll,
        (y_broadcasted, shift_grid),
        fn_output_signature=tf.TensorSpec(shape=(None, None, None, None, None, None), dtype=y.dtype)
    )

    # Transpose to shape (batch, N, N, N, channels, num_shifts, out_channels)
    rolled = tf.transpose(rolled, perm=[1, 2, 3, 4, 5, 0, 6])

    return rolled


In [66]:
y = tf.random.normal((2, 4, 4, 4, 5, 3))
delays = 1
# out_channels = 10
#5. Initialize y of shape (batch, N, N, channels, out_channels)
#6. Prepare circularly shifted y with delays, Y of (batch, N, N, channels, (D+1)^{\ensuremath{2}}, out_channels)

y_shifted = circular_shift_y_3d(y, delays)
print(y.shape, "->", y_shifted.shape)  # (8, 32, 32, 4, 9, 3)

# (2, 4, 4, 4, 5, 3) -> (2, 4, 4, 4, 5, 8, 3)

(2, 4, 4, 4, 5, 3) -> (2, 4, 4, 4, 5, 8, 3)


In [74]:

bs, inch, outch = 0, 0, 2

for axis3 in range(y.shape[3]):
    print(f"y axis3 {axis3}:=\n{y[bs,:,:,axis3,inch, outch]}\n")


for axis3 in range(y.shape[3]):
    for dela in range(y_shifted.shape[-2]):
        print(f"\ny axis3 {axis3} delay {dela}: \n{y_shifted[bs, :, :, axis3, inch, dela, outch]}")


y axis3 0:=
[[-0.7918891  -1.5366937   0.903857   -0.5792636 ]
 [-1.8596557   1.4587045   0.6052078  -0.7042012 ]
 [-0.82195795  0.8641602  -0.22416203  1.587956  ]
 [ 1.1599133  -1.7393975   1.8626752   1.6834183 ]]

y axis3 1:=
[[-0.26256844 -2.8363094  -0.9486382  -0.51092845]
 [ 1.1505169   0.7767548   0.29983628 -1.2667707 ]
 [ 0.64203405 -1.389705   -1.4869754   2.2604542 ]
 [ 0.6143161   0.22287488  1.5794418  -0.50284123]]

y axis3 2:=
[[ 0.18977298  0.24452719  0.82241786 -0.48539147]
 [ 0.95103097 -1.1060857  -0.624316    0.12974182]
 [ 0.7442143  -0.29222924  0.2816049  -0.88205916]
 [-0.5292267  -0.00533635  0.6192376  -0.79829186]]

y axis3 3:=
[[-0.00535059 -0.39806393 -0.58003134 -1.0866591 ]
 [ 0.8222045  -0.39105612  2.0373526  -0.06577711]
 [ 0.9049255   1.071593    0.36329132 -0.6413152 ]
 [-1.6068621  -0.98137873  0.821835   -0.05435105]]


y axis3 0 delay 0: 
[[-0.7918891  -1.5366937   0.903857   -0.5792636 ]
 [-1.8596557   1.4587045   0.6052078  -0.7042012 ]
 [-0.

# IIR 3D layer

         IIR 3D layer final

        circular shift dummy y of shape 
        (batch, N, inchannel, outchannel) -> (batch, N, inchannel, outchannel, Delays+1)

        circulal shift x
        (batch_size, N, inchannels) -> (batch_size, N, inchannels, Delays+1, outchannels)

In [ ]:
## OK general IIR 3D layer
import tensorflow as tf

@tf.keras.utils.register_keras_serializable()
class Positive(tf.keras.constraints.Constraint):
    """Makes sure non negative by clipping -ve values to 0
    
    License: GPLv3
    Copyright (C) Kishore K tarafdar
    """
    def __call__(self, w):
        return tf.maximum(w, 0.0)  # Clip negative values to 0
        # return tf.nn.relu(w)
        return w

    def get_config(self):
        return {}  # No parameters to serialize

# class No(Constraint):
#     def __call__(self, w):
#         # return tf.maximum(w, 0.0)  # Clip negative values to 0
#         # return tf.nn.relu(w)
#         return w

@tf.keras.utils.register_keras_serializable()
class IIR3D(tf.keras.layers.Layer):
    """IIRTF: Fast trainable multidimensional IIR filter layers in TensorFlow.
    Copyright (C) 2025 Kishore Kumar Tarafdar

    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, either version 3 of the License, or
    (at your option) any later version.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.

    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <https://www.gnu.org/licenses/>.
    
    IIR 3D Layer --kkt 24-05-2025"""
    def __init__(self, Delays:int, filters:int, tolerance=1e-6, max_steps=5000, local_lr=0.001, **kwargs):
        super().__init__(**kwargs)
        self.Delays = Delays
        self.filters = filters ## number of out channels
        self.tolerance = tolerance
        self.max_steps = max_steps
        self.local_lr = local_lr
        # Define optimizer (can be class attribute if needed)
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=self.local_lr)

    def build(self, input_shape):
        self.N = input_shape[1]
        self.channels = input_shape[-1]

        # Create grid of 3D shifts: (D+1, D+1, D+1, 3) → (num_shifts, 3)
        shifts = tf.range(self.Delays + 1)
        shift_grid = tf.stack(tf.meshgrid(shifts, shifts, shifts, indexing="ij"), axis=-1)  # (D+1, D+1, D+1, 3)
        self.shift_grid = tf.reshape(shift_grid, [-1, 3])  # (num_shifts, 3)
        
        ## IIR feed forward coeffs
        self.b = self.add_weight(
            shape=(self.channels, int((self.Delays + 1)**3), self.filters), 
            initializer=tf.keras.initializers.RandomUniform(minval=0.0, maxval=0.1), 
            trainable=True, 
            name="b_coeffs",
            constraint=Positive() ## no constraint
        )
        ## IIR feed back coeffs
        self.a = self.add_weight(
            shape=(self.channels, int((self.Delays + 1)**3-1), self.filters), 
            initializer=tf.keras.initializers.RandomUniform(minval=0.0, maxval=0.1), 
            trainable=True, 
            name="a_coeffs",
            constraint=Positive() ## constraint >0 for STABLE FILTER
        )
        super().build(input_shape)
        # ## Setting a0=1 in feedback coeffs
        # a0s = tf.ones((self.channels, 1, self.filters))
        # self.A = tf.concat([a0s, self.a], axis=-2)  # shape: (channels, Delays+1)
        # self.output_shape = (input_shape[0], input_shape[1], input_shape[2], input_shape[3], self.filters) 
       
    # @tf.function(jit_compile=False)  # Disable XLA for debugging
    def call(self, x):
        ## Setting a0=1 in feedback coeffs
        a0s = tf.ones((self.channels, 1, self.filters))
        A = tf.concat([a0s, self.a], axis=-2)  # shape: (channels, Delays+1)
        
        ## initializing IIR filter or layer output y
        # y = tf.identity(x)  # start from input
        # y = tf.zeros_like(x)
        # (batch, N, N, N, channels, out_channels)
        y = tf.random.uniform(
            shape=(tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3], tf.shape(x)[4], self.filters),
            minval=0,
            maxval=1
            )

        ## Creating shifted tensor: [x[n], x[n-1], ... , x[n-D]] 
        Xshifted = self.__circularshift_x3d(x)  # Precompute once
        # self.Xshifted = Xshifted

        ## Local optimization loss
        def compute_residual(y_var):
            """Compute Residual |AX-BY| for local optimization"""
            Yshifted = self.__circularshift_y3d(y_var)
            # print('+ Xshifted',Xshifted.shape, self.b.shape)
            # print('+ Yshifted',Yshifted.shape, self.a.shape)
            BX = tf.einsum('bnmlcdo,cdo->bnmlo', Xshifted, self.b)
            AY = tf.einsum('bnmlcdo,cdo->bnmlo', Yshifted, A)
            
            residual = BX - AY
            print(BX.shape, AY.shape, residual.shape)
            # loss = tf.reduce_sum(residual) # unstable filters
            # loss = tf.reduce_sum(tf.abs(residual)) #better
            loss = tf.reduce_sum(tf.square(residual)) #better
            return loss, residual

        # Optimization loop------
        i = tf.constant(0)
        loss_init, _ = compute_residual(y)
        
        # i = tf.constant(0, dtype=tf.int32)
        # loss_init = tf.constant(float('inf'), dtype=x.dtype)

        def cond(i, y_var, loss):
            # print(loss.numpy(), self.tolerance, (loss > self.tolerance).numpy())
            return tf.logical_and(loss > self.tolerance, i < self.max_steps)
        
        def body(i, y_var, loss):
            with tf.GradientTape() as tape:
                print(f'{i}', end=', ')
                tape.watch(y_var)
                loss, _ = compute_residual(y_var)
            grads = tape.gradient(loss, [y_var] + self.trainable_variables)
            dy = grads[0]
            # dvars = grads[1:]

            # Update y manually
            y_var = y_var - self.local_lr * dy
            return i + 1, y_var, loss

        i, y_opt, loss_final = tf.while_loop(
            cond, 
            body,
            loop_vars=[i, y, loss_init],
            # maximum_iterations=self.max_steps,  # Critical for XLA
            # shape_invariants=[
            #     i.shape,
            #     tf.TensorShape([None, self.N, self.channels]),  # Batch dimension can vary
            #     loss_init.shape
            # ]
            # maximum_iterations=self.max_steps
            # maximum_iterations=None
        )
        ## END local optimization----------
                
        y = tf.einsum('bnmlco->bnmlo', y_opt)
        # print('y shape' ,y.shape)
        # y = tf.expand_dims(, axis=-1)
        return y
 
    def __circularshift_x3d(self, x):
        """
        x: Tensor of shape (batch, N, N, N, channels)
        delays: int, number of delays in each dimension
        out_channels: int, number of output channels (identical copies)

        Returns:
            Tensor of shape (batch, N, N, N, channels, (D+1)^3, out_channels)
        """
        batch, N, _, _, channels = tf.unstack(tf.shape(x))
        out_channels = self.filters

        # Create grid of 3D shifts: (D+1, D+1, D+1, 3) → (num_shifts, 3)
        # shifts = tf.range(self.Delays + 1)
        # shift_grid = tf.stack(tf.meshgrid(shifts, shifts, shifts, indexing="ij"), axis=-1)  # (D+1, D+1, D+1, 3)
        # shift_grid = tf.reshape(shift_grid, [-1, 3])  # (num_shifts, 3)

        # Broadcast x to (num_shifts, batch, N, N, N, channels)
        x_broadcasted = tf.expand_dims(x, axis=0)
        x_broadcasted = tf.repeat(x_broadcasted, repeats=tf.shape(self.shift_grid)[0], axis=0)

        # Roll each slice using vectorized shift
        def single_roll(args):
            x_i, shift = args
            return tf.roll(x_i, shift=[-shift[0], -shift[1], -shift[2]], axis=[1, 2, 3])

        rolled = tf.map_fn(
            single_roll,
            (x_broadcasted, self.shift_grid),
            fn_output_signature=tf.TensorSpec(shape=(None, None, None, None, None), dtype=x.dtype)
        )

        # rolled shape: (num_shifts, batch, N, N, N, channels) → (batch, N, N, N, channels, num_shifts)
        rolled = tf.transpose(rolled, perm=[1, 2, 3, 4, 5, 0])

        # Expand and tile to add out_channels: (batch, N, N, N, channels, num_shifts, out_channels)
        rolled = tf.expand_dims(rolled, axis=-1)
        rolled = tf.tile(rolled, [1, 1, 1, 1, 1, 1, out_channels])

        return rolled

    def __circularshift_y3d(self, y):
        """
        y: Tensor of shape (batch, N, N, N, channels, out_channels)
        delays: int, number of delays in each dimension

        Returns:
            Tensor of shape (batch, N, N, N, channels, (D+1)^3, out_channels)
        """
        batch, N, _, _, channels, out_channels = tf.unstack(tf.shape(y))

        # Create grid of 3D shifts: (D+1, D+1, D+1, 3) → (num_shifts, 3)
        # shifts = tf.range(self.Delays + 1)
        # shift_grid = tf.stack(tf.meshgrid(shifts, shifts, shifts, indexing="ij"), axis=-1)  # (D+1, D+1, D+1, 3)
        # shift_grid = tf.reshape(shift_grid, [-1, 3])  # (num_shifts, 3)

        # Broadcast y to (num_shifts, batch, N, N, N, channels, out_channels)
        y_broadcasted = tf.expand_dims(y, axis=0)
        y_broadcasted = tf.repeat(y_broadcasted, repeats=tf.shape(self.shift_grid)[0], axis=0)

        # Roll each slice using vectorized shift
        def single_roll(args):
            y_i, shift = args
            return tf.roll(y_i, shift=[-shift[0], -shift[1], -shift[2]], axis=[1, 2, 3])

        rolled = tf.map_fn(
            single_roll,
            (y_broadcasted, self.shift_grid),
            fn_output_signature=tf.TensorSpec(shape=(None, None, None, None, None, None), dtype=y.dtype)
        )

        # Transpose to shape (batch, N, N, N, channels, num_shifts, out_channels)
        rolled = tf.transpose(rolled, perm=[1, 2, 3, 4, 5, 0, 6])

        return rolled

    def get_config(self):
        config = super().get_config()
        config.update({
            'Delays': self.Delays,
            'filters': self.filters,
            'tolerance': self.tolerance,
            'max_steps': self.max_steps,
            'local_lr': self.local_lr,
        })
        return config


if __name__=='__main__':
    # Example usage:
    batch_size = 2
    N = 16
    channels = 5
    filters = 3
    # x = tf.random.normal((batch_size, N, channels))
    x = tf.random.uniform(shape=(batch_size, N, N, N, channels), minval=0, maxval=1)
    print('x', x.shape)

    Delays = 1
    layer = IIR3D(Delays=Delays, filters=filters, max_steps=5)#N-1)
    out = layer(x)
    # out = layer(out)
    print("Residual output shape:", out.shape)
    print(out.numpy())


    ##Example 
    N = 16
    input_shape = (N, N, N, 12)
    inputs = tf.keras.Input(shape=input_shape)
    outputs = IIR3D(Delays=2, filters=4)(inputs)
    outputs = IIR3D(Delays=2, filters=2)(outputs)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.summary()
    del N, input_shape, inputs, outputs, model



x (2, 16, 16, 16, 5)
(2, 16, 16, 16, 3) (2, 16, 16, 16, 3) (2, 16, 16, 16, 3)
0, (2, 16, 16, 16, 3) (2, 16, 16, 16, 3) (2, 16, 16, 16, 3)
1, (2, 16, 16, 16, 3) (2, 16, 16, 16, 3) (2, 16, 16, 16, 3)
2, (2, 16, 16, 16, 3) (2, 16, 16, 16, 3) (2, 16, 16, 16, 3)
3, (2, 16, 16, 16, 3) (2, 16, 16, 16, 3) (2, 16, 16, 16, 3)
4, (2, 16, 16, 16, 3) (2, 16, 16, 16, 3) (2, 16, 16, 16, 3)
Residual output shape: (2, 16, 16, 16, 3)
[[[[[1.7756574  1.5678921  1.3550165 ]
    [2.6500158  2.167324   3.2112827 ]
    [2.335089   3.123149   2.0953984 ]
    ...
    [2.3560572  2.2681563  1.6848509 ]
    [2.0948606  2.207283   2.9460545 ]
    [2.7318017  2.6105094  1.8836133 ]]

   [[2.2859018  2.6915836  2.5910048 ]
    [1.8696485  2.035164   2.0973394 ]
    [2.7951322  2.0034473  2.073138  ]
    ...
    [2.9882214  2.592583   2.839819  ]
    [1.6524848  3.2106948  2.6605172 ]
    [3.097149   2.2820187  2.3466058 ]]

   [[3.0866182  1.4940809  2.646245  ]
    [1.9827056  1.9192237  1.5930148 ]
    [1.3759534

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 16, 16, 16, 12) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ iir3d_13 (IIR3D)                │ (None, 16, 16, 16, 4)  │         2,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ iir3d_14 (IIR3D)                │ (None, 16, 16, 16, 2)  │           424 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,968 (11.59 KB)

 Trainable params: 2,968 (11.59 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
out.shape

TensorShape([2, 16, 16, 16, 3])

In [6]:
layer.b.shape, layer.a.shape #layer.Xshifted.shape

(TensorShape([5, 8, 3]), TensorShape([5, 7, 3]))

In [7]:
out.shape

TensorShape([2, 16, 16, 16, 3])

        plot

In [8]:
out.shape, layer.a.shape, layer.b.shape, layer.b

(TensorShape([2, 16, 16, 16, 3]),
 TensorShape([5, 7, 3]),
 TensorShape([5, 8, 3]),
 <KerasVariable shape=(5, 8, 3), dtype=float32, path=iir3d_1/b_coeffs>)

In [9]:
for i in range(out.shape[-1]):
    print(out[0,:,i].numpy())

[[[1.8382825  2.23938    3.0693896 ]
  [3.1792407  3.5410788  2.620131  ]
  [2.4580617  2.3446276  1.939734  ]
  [2.8211129  2.8604732  2.8920798 ]
  [1.8248668  2.1653066  2.5663729 ]
  [1.05334    1.6740131  2.2700784 ]
  [2.7912235  1.9168622  2.370846  ]
  [2.8688848  2.6566942  1.90784   ]
  [2.369776   3.9845853  3.005563  ]
  [2.6373649  1.5308788  1.7195832 ]
  [3.0119977  1.204947   1.9447901 ]
  [2.1938467  2.6420622  0.94337565]
  [1.9375433  1.9959344  2.7872872 ]
  [1.9856639  3.8262086  2.8958628 ]
  [3.3010921  1.8954024  2.648159  ]
  [2.3157077  1.9366627  1.940964  ]]

 [[2.7702255  2.6054158  1.7828013 ]
  [2.6006815  3.3659518  1.6601484 ]
  [2.4845705  1.9880993  2.5954046 ]
  [2.4789388  2.2383373  2.604324  ]
  [1.9566457  2.629469   2.1404588 ]
  [2.8568842  3.2506886  2.6601787 ]
  [2.7857184  3.0841656  1.6401763 ]
  [2.370698   1.829652   3.3196375 ]
  [3.241112   2.8609076  2.2086396 ]
  [2.2118742  1.7391545  3.1452997 ]
  [1.9258845  2.6010242  2.3459594 ]

In [10]:
layer.a.numpy(), layer.b.numpy()

(array([[[0.03842386, 0.00896555, 0.01434171],
         [0.08398982, 0.04187762, 0.04910704],
         [0.04004646, 0.08737347, 0.07246944],
         [0.06780752, 0.09797911, 0.05454991],
         [0.00911272, 0.06110854, 0.09998389],
         [0.05882613, 0.08117386, 0.05213562],
         [0.06377983, 0.09427279, 0.04068786]],
 
        [[0.02781417, 0.08407476, 0.03527058],
         [0.07945184, 0.04808096, 0.01100988],
         [0.07550006, 0.06408347, 0.09197469],
         [0.00246116, 0.01051396, 0.08926832],
         [0.03201556, 0.0590478 , 0.02838677],
         [0.03708413, 0.05702681, 0.0584478 ],
         [0.00431769, 0.00283084, 0.00448588]],
 
        [[0.04974678, 0.08434188, 0.08508464],
         [0.0845606 , 0.04774165, 0.05688713],
         [0.09208788, 0.07812209, 0.07403677],
         [0.02660319, 0.05864396, 0.0267476 ],
         [0.01754883, 0.00557605, 0.06127654],
         [0.06510901, 0.01061097, 0.05307839],
         [0.00984783, 0.01231515, 0.03117952]],
 
    

In [11]:
layer.a.shape, layer.b.shape

(TensorShape([5, 7, 3]), TensorShape([5, 8, 3]))

In [12]:
ch = 1
filt = 0

btmp = layer.b[ch,:,filt].numpy()
atmp = layer.a[ch,:,filt].numpy()
btmp, atmp
# plot_pole_zero(btmp, atmp)

(array([0.05191571, 0.08958735, 0.03897641, 0.051667  , 0.07407693,
        0.0098083 , 0.0113165 , 0.00425371], dtype=float32),
 array([0.02781417, 0.07945184, 0.07550006, 0.00246116, 0.03201556,
        0.03708413, 0.00431769], dtype=float32))

In [13]:
layer.b.numpy()#, layer.a

array([[[0.04774748, 0.08898009, 0.09877353],
        [0.02940064, 0.08198849, 0.04031542],
        [0.03746003, 0.00693835, 0.09485825],
        [0.03695707, 0.0674622 , 0.03936642],
        [0.08266073, 0.01940926, 0.09618732],
        [0.07808591, 0.02850934, 0.07815105],
        [0.08915202, 0.06133121, 0.00936272],
        [0.0088606 , 0.04407135, 0.05132888]],

       [[0.05191571, 0.08103323, 0.02174387],
        [0.08958735, 0.0396695 , 0.01294627],
        [0.03897641, 0.06993522, 0.01910839],
        [0.051667  , 0.04278213, 0.08669479],
        [0.07407693, 0.07125673, 0.04107377],
        [0.0098083 , 0.02182063, 0.08304985],
        [0.0113165 , 0.08335225, 0.09790144],
        [0.00425371, 0.03958163, 0.04135216]],

       [[0.05863469, 0.07618632, 0.09822284],
        [0.0175681 , 0.01723838, 0.09048954],
        [0.08871263, 0.05953234, 0.06834202],
        [0.01805502, 0.09348545, 0.04213605],
        [0.01287233, 0.06386242, 0.09249915],
        [0.05633603, 0.0562687

In [31]:
# # for j in range(bs):
# for filt in range(filters):
#     for ch in range(channels):  
#         # Plot for first batch and first channel
#         b_example = layer.b[ch,:,filt].numpy()
#         a_example = layer.a[ch,:,filt].numpy()
#         print(f"filter: {filt}, channel: {ch},\ncoeffs b: {b_example}\ncoeffs a: {a_example}")
#         plot_pole_zero(b_example, a_example)

# # for j in range(bs):
# for filt in range(filters):
#     for ch in range(channels):
#         # Plot for first batch and first channel
#         b_example = layer.b[ch,:,filt].numpy()
#         a_example = layer.a[ch,:,filt].numpy()
#         print(f"filter: {filt}, channel: {ch},\ncoeffs b: {b_example}\ncoeffs a: {a_example}")
#         plot_frequency_response(b_example, a_example)


In [20]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

# Defining a model with IIR3D layer and Training

        Define an model with the IIR3D layer and check

In [14]:
# Dummy data
batch_size, N, channels = 2, 8, 9
x_train = tf.random.normal((batch_size, N, N, N, channels))
y_dummy = tf.random.normal((batch_size, N, N, N, filters))
# y_dummy = tf.zeros_like(x_train)
x_train.shape, y_dummy.shape, x_train.dtype, y_dummy.dtype
# compile model
# model = Layer1D(D=Delays,max_steps=100)#N-1)
# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01))
# Build and compile model
# layer = Layer1D(D=N-1)


# def model(input_shape):
#     inputs = tf.keras.Input(shape=input_shape)  # shape: (batch, N, channels)
#     x = Layer1D(D=100)(inputs)        # shape: (batch, N, channels)
#     outputs = tf.keras.layers.Dense(channels)(x)  # project back to same shape
#     return tf.keras.Model(inputs=inputs, outputs=outputs)


# # # Fit model for 2 epochs
# # model.fit(x=x_train, y=y_dummy, epochs=2, batch_size=10)

(TensorShape([2, 8, 8, 8, 9]),
 TensorShape([2, 8, 8, 8, 3]),
 tf.float32,
 tf.float32)

In [15]:
Delays, filters

(1, 3)

In [16]:
def model3d(
    input_shape=(32,32,32, 1),
    # Delays=Delays,
    # filters=filters
    # loss = 'binary_crossentropy',
    # optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    # compile=True
    ):

    inputs = tf.keras.layers.Input(input_shape)
    q = IIR3D(Delays=2, filters=6, tolerance=1e-6, max_steps=100, local_lr=0.001,)(inputs)
    # print('q', q.shape)
    q = IIR3D(Delays=2, filters=3, tolerance=1e-6, max_steps=100, local_lr=0.001,)(q)
    
    
    outputs = q  
    model = tf.keras.Model(inputs=[inputs], outputs=[outputs])
    return model

    # if compile==True:
    #     # model = tf.keras.Model(inputs=[inputs], outputs=[x])
    #     # generator_optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4)
    #     # model.compile(optimizer=generator_optimizer, loss=generator_loss)
       
    #     model.compile(optimizer=optimizer, loss=loss)
    #     return model
    #     #
    # else: 
    #     return model


m = model3d(input_shape=(N, N, N, channels))#, Delays=Delays, filters=filters)  # (N=20, channels=16)
# m.compile(optimizer='adam', loss='mse')

# model = LayerModel(layer)
m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse', jit_compile=False)  # Disable XLA for debugging)
m.summary()



(None, 8, 8, 8, 6) (None, 8, 8, 8, 6) (None, 8, 8, 8, 6)
Tensor("while/Placeholder:0", shape=(), dtype=int32), (None, 8, 8, 8, 6) (None, 8, 8, 8, 6) (None, 8, 8, 8, 6)
(None, 8, 8, 8, 3) (None, 8, 8, 8, 3) (None, 8, 8, 8, 3)
Tensor("while/Placeholder:0", shape=(), dtype=int32), (None, 8, 8, 8, 3) (None, 8, 8, 8, 3) (None, 8, 8, 8, 3)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 8, 8, 8, 9)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ iir3d_4 (IIR3D)                 │ (None, 8, 8, 8, 6)     │         2,862 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ iir3d_5 (IIR3D)                 │ (None, 8, 8, 8, 3)     │           954 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,816 (14.91 KB)

 Trainable params: 3,816 (14.91 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Run for 2 epochs
m.fit(x=x_train, y=y_dummy, epochs=10, batch_size=batch_size)

Epoch 1/10


/data1/kishoretarafdar/miniforge3/envs/tf218/lib/python3.12/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor_3']. Received: the structure of inputs=*
  warnings.warn(


(2, 8, 8, 8, 6) (2, 8, 8, 8, 6) (2, 8, 8, 8, 6)
Tensor("functional_1_1/iir3d_4_1/while/Placeholder:0", shape=(), dtype=int32), (2, 8, 8, 8, 6) (2, 8, 8, 8, 6) (2, 8, 8, 8, 6)
(2, 8, 8, 8, 3) (2, 8, 8, 8, 3) (2, 8, 8, 8, 3)
Tensor("functional_1_1/iir3d_5_1/while/Placeholder:0", shape=(), dtype=int32), (2, 8, 8, 8, 3) (2, 8, 8, 8, 3) (2, 8, 8, 8, 3)
(2, 8, 8, 8, 6) (2, 8, 8, 8, 6) (2, 8, 8, 8, 6)
Tensor("functional_1_1/iir3d_4_1/while/Placeholder:0", shape=(), dtype=int32), (2, 8, 8, 8, 6) (2, 8, 8, 8, 6) (2, 8, 8, 8, 6)
(2, 8, 8, 8, 3) (2, 8, 8, 8, 3) (2, 8, 8, 8, 3)
Tensor("functional_1_1/iir3d_5_1/while/Placeholder:0", shape=(), dtype=int32), (2, 8, 8, 8, 3) (2, 8, 8, 8, 3) (2, 8, 8, 8, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 85s 85s/step - loss: 2.1622
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 79s 79s/step - loss: 1.4451
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 78s 78s/step - loss: 1.2313
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 78s 78s/step - loss: 1.1236
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 77s 77s/step - loss

        Troubleshoot gradient update

In [11]:
# with tf.GradientTape() as tape:
#     y = m(x_train)
#     loss = tf.reduce_mean(tf.square(y - y_dummy))

# grads = tape.gradient(loss, m.trainable_variables)
# for var, g in zip(m.trainable_variables, grads):
#     print(var.name, "grad:", g)

# sample_x = x_train[:1]
# sample_y = y_train[:1]

with tf.GradientTape() as tape:
    y_pred = m(x_train)
    loss = tf.reduce_mean(tf.square(y_pred - y_dummy))

grads = tape.gradient(loss, m.trainable_variables)
for var, g in zip(m.trainable_variables, grads):
    print(f"{var.name}: {'OK' if g is not None else '❌ No Grad'}")


0, 1, 2, 

3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, b_coeffs: OK
a_coeffs: OK
b_coeffs: OK
a_coeffs: OK


In [ ]:
with tf.GradientTape() as tape:
    y_pred = m(x_train)
    loss = tf.reduce_mean(tf.square(y_pred - y_dummy))

grads = tape.gradient(loss, m.trainable_variables)
for var, g in zip(m.trainable_variables, grads):
    print(f"{var.name}: {'✅ OK' if g is not None else '❌ No Grad'}")

for var, grad in zip(m.trainable_variables, grad):
    if grad is not None:
        print(f"{var.name}: mean gradient {tf.reduce_mean(tf.abs(grad)):.6f}")
    else:
        print(f"{var.name}: NO GRADIENTS")


In [26]:
m.predict(x_train).shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step, dtype=int32), Tensor("functional_1_1/iir_layer1d_4_1/while/Placeholder:0", shape=(), dtype=int32),


(10, 128, 3)

In [27]:
[_.shape for _ in m.layers[1].get_weights()]
# [_.shape for _ in m.layers[2].get_weights()]
# [_.shape for _ in m.layers[3].get_weights()]


[(12, 41, 10), (12, 40, 10)]

In [28]:
[_.shape for _ in m.weights]

[TensorShape([12, 41, 10]),
 TensorShape([12, 40, 10]),
 TensorShape([10, 41, 3]),
 TensorShape([10, 40, 3])]

In [29]:
m.weights[0].numpy()
m.weights[1].numpy().shape

(12, 40, 10)

In [3]:
b = m.weights[0].numpy()
a = m.weights[1].numpy()
# b = m.weights[2].numpy()
# a = m.weights[3].numpy()

a0s = tf.ones((channels, 1, filters))
a.shape, a0s.shape
# a = tf.concat([a0s, a], axis=-2)  # shape: (channels, Delays+1, outchannels)
# a0s.shape, a.shape, b.shape

NameError: name 'm' is not defined

In [ ]:
# # for j in range(bs):
# for filt in range(filters):
#     for ch in range(channels):
#         # Plot for first batch and first channel
#         b_example = b[ch,:,filt]
#         a_example = a[ch,:,filt]
#         print(f"filter: {filt}, channel: {ch},\ncoeffs b: {b_example}\ncoeffs a: {a_example}")
#         plot_pole_zero(b_example, a_example)


In [ ]:
# for filt in range(filters):
#     for ch in range(channels):
#         # Plot for first batch and first channel
#         b_example = b[ch,:,filt]
#         a_example = a[ch,:,filt]
#         print(f"filter: {filt}, channel: {ch},\ncoeffs b: {b_example}\ncoeffs a: {a_example}")
#         plot_frequency_response(b_example, a_example)